# 1. Dataset Preparation & Exploration

Safespace AI utilizes multiple datasets. Use this notebook to download datasets from Roboflow, explore their structure, and verify YOLO formatting.

In [ ]:
# Cross-Platform Environment Setup
import os
import sys

def setup_environment():
    """Detects platform and sets up the environment."""
    in_colab = 'google.colab' in sys.modules
    in_kaggle = os.environ.get('KAGGLE_URL_BASE') is not None
    
    if in_colab or in_kaggle:
        print(f"Running on {'Google Colab' if in_colab else 'Kaggle'}. Installing dependencies...")
        !pip install -qU ultralytics wandb roboflow python-dotenv supervision easyocr cvzone
        
        if in_colab:
            from google.colab import userdata
            os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY') or ''
            os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY') or ''
        elif in_kaggle:
            try:
                from kaggle_secrets import UserSecretsClient
                user_secrets = UserSecretsClient()
                os.environ['WANDB_API_KEY'] = user_secrets.get_secret('WANDB_API_KEY')
                os.environ['ROBOFLOW_API_KEY'] = user_secrets.get_secret('ROBOFLOW_API_KEY')
            except Exception:
                print("Kaggle secrets not found. Please set them in the Add-ons menu.")
    else:
        print("Running locally. Loading environment...")
        try:
            from dotenv import load_dotenv
            load_dotenv('../.env')
        except ImportError:
            print("python-dotenv not found. Install it with: pip install python-dotenv")

    # Verify keys
    if not os.environ.get('WANDB_API_KEY'):
        print("Warning: WANDB_API_KEY not set.")
    if not os.environ.get('ROBOFLOW_API_KEY'):
        print("Warning: ROBOFLOW_API_KEY not set.")

setup_environment()

## 2. Download Accident Detection Dataset

In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY"))
project = rf.workspace("zihan-yv8sc").project("zihan.v5i.yolov8-dataset")
version = project.version(5)

dataset_dir = '../datasets'
os.makedirs(dataset_dir, exist_ok=True)

dataset = version.download("yolov8", location=dataset_dir)

## 3. Download License Plate Recognition Dataset

In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY"))
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(11)

dataset_dir = '../datasets'
os.makedirs(dataset_dir, exist_ok=True)

dataset = version.download("yolov8", location=dataset_dir)

## 4. Explore Datasets

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image

def check_yolo_dataset(dataset_path):
    print(f'\n--- Checking: {os.path.basename(dataset_path.strip("/\\ "))} ---')
    
    # Check for Roboflow export patterns
    paths_to_check = [
        ('Standard', os.path.join(dataset_path, 'train', 'images')),
        ('Alternative', os.path.join(dataset_path, 'images', 'train'))
    ]
    
    found = False
    for label, path in paths_to_check:
        if os.path.exists(path):
            num_images = len(os.listdir(path))
            print(f'✅ Structure OK ({label})')
            print(f'Train Images: {num_images}')
            found = True
            break
            
    if not found:
        print('⚠️ Dataset structure incomplete or paths incorrect.')

datasets = glob.glob('../datasets/*/')
for d in datasets:
    if os.path.exists(os.path.join(d, 'data.yaml')):
        check_yolo_dataset(d)